# 02c — Feature Engineering (Setup C: Feb-Apr 2026 Target)

**Cross-period robustness companion to `02_FeatureEngineering.ipynb`** (Setup B: 6-month
history Jul-Dec 2025 -> target Q1 2026). This notebook builds **Setup C**: the *deepest*-history
configuration, used only to check whether the modelling approach generalizes to a different
(and later) quarter, not as a primary analysis.

- **History**: Jul 2025 - Jan 2026 (7 months -- Setup B's 6 months + January 2026)
- **Base listings snapshot**: January 2026 (last month of history, immediately preceding the
  target quarter)
- **Target**: Feb-Apr 2026 (89 days: 28+31+30, 2026 is not a leap year), built with the
  identical clean multi-snapshot rule as Setup B, re-pointed at this setup's own 6 snapshots
  (Nov 2025, Dec 2025, Jan 2026 = last-3-history + Feb, Mar, Apr 2026 = target months)

**What changes vs. Setup B:** the existing Q3 (Jul-Sep)/Q4 (Oct-Dec) 2025 split and price panel
(Jul-Nov 2025, unchanged -- Dec/Jan price is also empty per `01_EDA.ipynb` Table 1) carry over
as-is. What's new: a 7th monthly block (`m1_*`, January 2026) alongside the existing
`m7`...`m12`, the combined-history block extended to all 7 months (renamed `hist_*`, not `h2_*`,
since "H2" specifically means Jul-Dec and no longer describes a window that now includes
January), and the trajectory features' recent/older split recomputed over 7 points instead of 6
(recent3 = Nov,Dec,Jan; older3 = Jul,Aug,Sep; Oct is the unused middle point).

**Target column name**: kept as `blocked_days_Q1_2026` (same literal name as Setup A/B) so
`16_Cross_Period_Robustness.ipynb` can reuse identical column-reference code across all three
setups' files, which are never mixed together.

**Outputs:** `outputs/train_local_setupC.parquet`, `outputs/test_local_setupC.parquet`,
`outputs/all_features_setupC.parquet`, `outputs/target_naive_setupC.parquet`.

## 1. Setup

In [1]:
import ast
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('NumPy:', np.__version__, '| Pandas:', pd.__version__)

NumPy: 2.4.6 | Pandas: 3.0.3


## 2. Paths & Config (Setup C: Jul 2025-Jan 2026 history -> Feb-Apr 2026 target)

In [2]:
BASE_DIR = Path('../AirBnb_Inside')
Q3_DIR   = BASE_DIR / '2025_Inside_Airbnb/Q3'
Q4_DIR   = BASE_DIR / '2025_Inside_Airbnb/Q4'
Q1_DIR   = BASE_DIR / '2026_Inside_Airbnb'
DEC_DIR  = Q4_DIR / 'December2025'
NOV_DIR  = Q4_DIR / 'November2025'
JAN_DIR  = Q1_DIR / 'January2026'   # base listings snapshot for Setup C
OUT_DIR  = Path('../outputs')
OUT_DIR.mkdir(exist_ok=True, parents=True)

# Setup B's 6 months (Jul-Dec 2025) plus a 7th: January 2026. Jan 2026's own scrape starts
# 2026-01-15 (confirmed on disk), so it only contributes ~17 days to the history window.
MONTH_CONFIGS = [
    ('Jul', Q3_DIR / 'July2025'      / 'calendar.csv', '2025-07-01', '2025-07-31'),
    ('Aug', Q3_DIR / 'August2025'    / 'calendar.csv', '2025-08-02', '2025-08-31'),
    ('Sep', Q3_DIR / 'September2025' / 'calendar.csv', '2025-09-02', '2025-09-30'),
    ('Oct', Q4_DIR / 'October2025'   / 'calendar.csv', '2025-10-01', '2025-10-31'),
    ('Nov', Q4_DIR / 'November2025'  / 'calendar.csv', '2025-11-02', '2025-11-30'),
    ('Dec', Q4_DIR / 'December2025'  / 'calendar.csv', '2025-12-04', '2025-12-31'),
    ('Jan', JAN_DIR                  / 'calendar.csv', '2026-01-15', '2026-01-31'),
]

print('Data root:', BASE_DIR.resolve())
print('Output  :', OUT_DIR.resolve())

Data root: /Users/bashkal/Desktop/ML/ML-Final/Internship/AirBnb_Inside
Output  : /Users/bashkal/Desktop/ML/ML-Final/Internship/outputs


In [3]:
def detailed_listings_path(month_dir):
    """Return path of the detailed (79+ col) listings file for a month directory."""
    for fname in ['listings.csv', 'listings-2.csv']:
        p = Path(month_dir) / fname
        with open(p) as f:
            ncols = len(f.readline().split(','))
        if ncols > 30:
            return p
    raise FileNotFoundError(f'No detailed listings file in {month_dir}')

## 3. Calendar Features (Jul 2025-Jan 2026, 7 months)

In [4]:
def load_calendar_month(cal_path, start, end):
    df = pd.read_csv(
        cal_path,
        usecols=['listing_id', 'date', 'available'],
        dtype={'listing_id': 'int64', 'available': 'category'},
    )
    df['date'] = pd.to_datetime(df['date'])
    return df[(df['date'] >= start) & (df['date'] <= end)].copy()

print(f'{"Month":<5} {"Rows":>10} {"Listings":>10} {"Block rate":>12}')
print('-' * 42)
parts = []
for label, cal_path, start, end in MONTH_CONFIGS:
    m_df = load_calendar_month(cal_path, start, end)
    m_df['month_num'] = pd.to_datetime(start).month
    parts.append(m_df)
    br = (m_df['available'] == 'f').mean()
    print(f'{label:<5} {len(m_df):>10,} {m_df["listing_id"].nunique():>10,} {br:>12.3f}')

hist_cal = pd.concat(parts, ignore_index=True)
del parts
print(f'\nHistory combined: {len(hist_cal):,} rows | {hist_cal["listing_id"].nunique():,} unique listings')

Month       Rows   Listings   Block rate
------------------------------------------


Jul    1,093,964     36,345        0.702


Aug    1,080,922     36,403        0.660


Sep    1,047,224     36,178        0.708


Oct    1,085,967     36,111        0.709


Nov    1,045,723     36,353        0.670


Dec      982,765     36,261        0.649


Jan      602,811     36,286        0.602

History combined: 6,939,376 rows | 41,277 unique listings


In [5]:
def agg_calendar(cal_df, prefix):
    """Per-listing calendar feature aggregation for a given time window."""
    cal = cal_df[['listing_id', 'date', 'available']].copy()
    cal['is_blocked'] = (cal['available'] == 'f').astype('int8')
    cal['is_avail']   = (cal['available'] == 't').astype('int8')
    cal['is_weekend'] = cal['date'].dt.dayofweek.isin([4, 5]).astype('int8')  # Fri + Sat

    core = cal.groupby('listing_id').agg(
        days_total   = ('is_blocked', 'count'),
        days_blocked = ('is_blocked', 'sum'),
        days_avail   = ('is_avail',   'sum'),
    )
    core['blocking_rate'] = core['days_blocked'] / core['days_total']
    core['avail_rate']    = core['days_avail']   / core['days_total']

    we = cal[cal['is_weekend'] == 1].groupby('listing_id')['is_blocked'].agg(
        we_days='count', we_blocked='sum')
    wd = cal[cal['is_weekend'] == 0].groupby('listing_id')['is_blocked'].agg(
        wd_days='count', wd_blocked='sum')
    we['we_blocking_rate'] = we['we_blocked'] / we['we_days'].replace(0, np.nan)
    wd['wd_blocking_rate'] = wd['wd_blocked'] / wd['wd_days'].replace(0, np.nan)

    cal_s = cal.sort_values(['listing_id', 'date'])
    cal_s['prev_avail']     = cal_s.groupby('listing_id')['is_avail'].shift(1)
    cal_s['new_blocking']   = ((cal_s['is_avail'] == 0) & (cal_s['prev_avail'] == 1)).astype('int8')
    cal_s['new_unblocking'] = ((cal_s['is_avail'] == 1) & (cal_s['prev_avail'] == 0)).astype('int8')
    trans = cal_s.groupby('listing_id').agg(
        new_blockings   = ('new_blocking',   'sum'),
        new_unblockings = ('new_unblocking', 'sum'),
    )

    result = (core
              .join(we[['we_blocking_rate', 'we_days']])
              .join(wd[['wd_blocking_rate', 'wd_days']])
              .join(trans))
    result['we_minus_wd_blocking'] = result['we_blocking_rate'] - result['wd_blocking_rate']

    result.columns = [f'{prefix}_{c}' for c in result.columns]
    result.index.name = 'listing_id'
    return result.reset_index()

In [6]:
q3_mask = hist_cal['date'] < '2025-10-01'
q4_mask = (hist_cal['date'] >= '2025-10-01') & (hist_cal['date'] < '2026-01-01')

print('Computing Q3 2025 aggregates (Jul-Sep)...')
agg_q3 = agg_calendar(hist_cal[q3_mask], 'q3')
print(f'  Shape: {agg_q3.shape}')

print('Computing Q4 2025 aggregates (Oct-Dec)...')
agg_q4 = agg_calendar(hist_cal[q4_mask], 'q4')
print(f'  Shape: {agg_q4.shape}')

print('Computing full 7-month combined aggregates (Jul 2025-Jan 2026)...')
agg_hist = agg_calendar(hist_cal, 'hist')
print(f'  Shape: {agg_hist.shape}')

Computing Q3 2025 aggregates (Jul-Sep)...


  Shape: (38203, 13)
Computing Q4 2025 aggregates (Oct-Dec)...


  Shape: (38077, 13)
Computing full 7-month combined aggregates (Jul 2025-Jan 2026)...


  Shape: (41277, 13)


In [7]:
# Monthly blocking rates M7-M12 (2025) + M1 (Jan 2026)
print('Computing monthly blocking rates M7-M12, M1...')
monthly_frames = []
for month_num in list(range(7, 13)) + [1]:
    mdata = hist_cal[hist_cal['month_num'] == month_num].copy()
    if len(mdata) == 0:
        print(f'  M{month_num}: no data (minor scrape gap)')
        continue
    m_agg = (
        mdata.assign(is_blocked=(mdata['available'] == 'f').astype('int8'))
        .groupby('listing_id')
        .agg(**{
            f'm{month_num}_blocking_rate': ('is_blocked', 'mean'),
            f'm{month_num}_days':          ('is_blocked', 'count'),
        })
    )
    monthly_frames.append(m_agg)
    print(f'  M{month_num}: {len(m_agg):,} listings | mean_br={m_agg[f"m{month_num}_blocking_rate"].mean():.3f}')

agg_monthly = monthly_frames[0]
for mf in monthly_frames[1:]:
    agg_monthly = agg_monthly.join(mf, how='outer')
agg_monthly = agg_monthly.reset_index()
print(f'\nMonthly feature table: {agg_monthly.shape}')

Computing monthly blocking rates M7-M12, M1...
  M7: 36,345 listings | mean_br=0.703
  M8: 36,403 listings | mean_br=0.660
  M9: 36,178 listings | mean_br=0.708


  M10: 36,111 listings | mean_br=0.710
  M11: 36,353 listings | mean_br=0.672
  M12: 36,261 listings | mean_br=0.650
  M1: 36,286 listings | mean_br=0.605

Monthly feature table: (41277, 15)


In [8]:
# Recency window: whatever January 2026 rows exist in hist_cal (scrape starts Jan 15,
# so this naturally captures Jan 15-31, ~17 days -- the most recent scraped window).
recent_mask = hist_cal['date'] >= '2026-01-01'
print('Computing recency window (January 2026)...')
agg_recent = agg_calendar(hist_cal[recent_mask], 'recent')
print(f'Shape: {agg_recent.shape}')

del hist_cal
print('hist_cal freed from memory.')

Computing recency window (January 2026)...


Shape: (36286, 13)
hist_cal freed from memory.


In [9]:
# Cross-snapshot volatility over the full 7-month history. Prefixed 'hist_' (not 'h2_' as
# in Setup B) since this window now extends past H2 2025 into January 2026.
VOL_SNAPS = [
    (1, Q3_DIR / 'July2025'      / 'calendar.csv'),
    (2, Q3_DIR / 'August2025'    / 'calendar.csv'),
    (3, Q3_DIR / 'September2025' / 'calendar.csv'),
    (4, Q4_DIR / 'October2025'   / 'calendar.csv'),
    (5, Q4_DIR / 'November2025'  / 'calendar.csv'),
    (6, DEC_DIR                  / 'calendar.csv'),
    (7, JAN_DIR                  / 'calendar.csv'),
]
VOL_START, VOL_END = '2025-07-01', '2026-01-31'

print('Building Jul 2025-Jan 2026 cross-snapshot volatility panel...')
vol_frames = []
for order, cal_path in VOL_SNAPS:
    df = pd.read_csv(cal_path, usecols=['listing_id', 'date', 'available'],
                      dtype={'listing_id': 'int64', 'available': 'category'})
    df = df[(df['date'] >= VOL_START) & (df['date'] <= VOL_END)].copy()
    df['order'] = np.int8(order)
    df['av'] = (df['available'] == 't')
    vol_frames.append(df[['listing_id', 'date', 'order', 'av']])
    print(f'  snapshot {order} ({cal_path.parent.name}): {len(df):,} history-window rows')

vol_panel = pd.concat(vol_frames, ignore_index=True).sort_values(['listing_id', 'date', 'order'])
del vol_frames

vol_panel['prev_av'] = vol_panel.groupby(['listing_id', 'date'])['av'].shift(1)
vol_panel['flipped']  = (vol_panel['prev_av'].notna() & (vol_panel['av'] != vol_panel['prev_av'])).astype('int8')
g = vol_panel.groupby(['listing_id', 'date'])['av']
multi_obs = (g.transform('size') >= 2).astype('int8')
vol_panel['multi_obs'] = multi_obs

agg_vol = vol_panel.groupby('listing_id').agg(
    hist_flips          = ('flipped',   'sum'),
    hist_multi_obs_days = ('multi_obs', 'sum'),
).reset_index()
agg_vol['hist_flip_rate'] = agg_vol['hist_flips'] / agg_vol['hist_multi_obs_days'].replace(0, np.nan)
agg_vol['id'] = agg_vol['listing_id'].astype(str)
del vol_panel

print(f'\nVolatility feature table: {agg_vol.shape}')
print(agg_vol[['hist_flips', 'hist_multi_obs_days', 'hist_flip_rate']].describe().round(3).to_string())

Building Jul 2025-Jan 2026 cross-snapshot volatility panel...


  snapshot 1 (July2025): 7,781,444 history-window rows


  snapshot 2 (August2025): 6,650,581 history-window rows


  snapshot 3 (September2025): 5,497,118 history-window rows


  snapshot 4 (October2025): 4,408,179 history-window rows


  snapshot 5 (November2025): 3,299,609 history-window rows


  snapshot 6 (December2025): 2,106,856 history-window rows


  snapshot 7 (January2026): 602,811 history-window rows



Volatility feature table: (41277, 5)
       hist_flips  hist_multi_obs_days  hist_flip_rate
count   41277.000            41277.000       39041.000
mean       48.814              698.692           0.074
std        65.571              237.088           0.095
min         0.000                0.000           0.000
25%         0.000              802.000           0.000
50%         9.000              805.000           0.022
75%        90.000              806.000           0.133
max       520.000              808.000           0.644


## 4. Property Metadata (January 2026 -- base snapshot for Setup C)

In [10]:
jan_file = detailed_listings_path(JAN_DIR)
print(f'Loading January 2026 listings: {jan_file.name}')
meta = pd.read_csv(jan_file, low_memory=False)
meta['id'] = meta['id'].astype(str)
print(f'Shape: {meta.shape}')

Loading January 2026 listings: listings.csv


Shape: (36282, 85)


In [11]:
# Multi-month price: unchanged from Setup B -- Jul-Nov 2025 (5 months). December 2025
# AND January 2026 listings price are both 100% empty (confirmed in 01_EDA.ipynb Table 1),
# so extending the price window into the new history months adds nothing.
PRICE_MONTHS = [
    ('07', Q3_DIR / 'July2025'),
    ('08', Q3_DIR / 'August2025'),
    ('09', Q3_DIR / 'September2025'),
    ('10', Q4_DIR / 'October2025'),
    ('11', NOV_DIR),
]

def parse_price_str(s):
    if pd.isna(s) or str(s).strip() in ('', 'nan'): return np.nan
    return float(str(s).replace('$', '').replace(',', '').strip())

print('Loading price across 5 months (Jul-Nov 2025)...')
price_series = {}
for m, month_dir in PRICE_MONTHS:
    f = detailed_listings_path(month_dir)
    d = pd.read_csv(f, usecols=['id', 'price'], dtype=str, low_memory=False)
    d['id'] = d['id'].astype(str)
    s = d.set_index('id')['price'].apply(parse_price_str)
    price_series[m] = s
    print(f'  {month_dir.name}: {s.notna().mean():.1%} price coverage')

price_panel = pd.concat(price_series, axis=1)
price_panel.columns = [f'p_{m}' for m in price_panel.columns]
pcols = list(price_panel.columns)
price_panel['p5_mean']  = price_panel[pcols].mean(axis=1)
price_panel['p5_std']   = price_panel[pcols].std(axis=1)
price_panel['p5_cv']    = price_panel['p5_std'] / price_panel['p5_mean'].replace(0, np.nan)
price_panel['p5_trend'] = (price_panel['p_11'] - price_panel['p_07']) / price_panel['p_07'].replace(0, np.nan)
price_panel['p5_min']   = price_panel[pcols].min(axis=1)
price_panel['p5_max']   = price_panel[pcols].max(axis=1)
price_panel['p5_range'] = price_panel['p5_max'] - price_panel['p5_min']
price_panel['p5_n_distinct'] = price_panel[pcols].nunique(axis=1, dropna=True).astype('int8')
price_panel = price_panel.reset_index().rename(columns={'index': 'id'})
print(f'\n5-month combined coverage: {price_panel["p5_mean"].notna().mean():.1%}  '
      f'(vs. single-month November: {price_panel["p_11"].notna().mean():.1%})')

meta = meta.merge(
    price_panel[['id', 'p5_mean', 'p5_std', 'p5_cv', 'p5_trend', 'p5_min', 'p5_max', 'p5_range', 'p5_n_distinct']],
    on='id', how='left',
)

meta['price_implied'] = (
    pd.to_numeric(meta.get('estimated_revenue_l365d'), errors='coerce') /
    pd.to_numeric(meta.get('estimated_occupancy_l365d'), errors='coerce').replace(0, np.nan)
)
meta['price_numeric'] = meta['p5_mean'].fillna(meta['price_implied'])
meta['price_missing'] = meta['p5_mean'].isna().astype('int8')

if 'neighbourhood_cleansed' in meta.columns:
    nb_median = meta.groupby('neighbourhood_cleansed')['p5_mean'].transform('median')
    meta['price_rel_nbhd'] = meta['p5_mean'] / nb_median.replace(0, np.nan)

print(f'\nFinal price_numeric coverage : {meta["price_numeric"].notna().mean():.1%}')
print(f'price_rel_nbhd coverage      : {meta["price_rel_nbhd"].notna().mean():.1%}')

Loading price across 5 months (Jul-Nov 2025)...


  July2025: 58.7% price coverage


  August2025: 58.5% price coverage


  September2025: 58.4% price coverage


  October2025: 59.1% price coverage


  November2025: 58.9% price coverage



5-month combined coverage: 67.5%  (vs. single-month November: 54.1%)

Final price_numeric coverage : 62.1%
price_rel_nbhd coverage      : 62.1%


In [12]:
for col in ['host_response_rate', 'host_acceptance_rate']:
    if col in meta.columns:
        meta[f'{col}_num'] = (
            meta[col].astype(str)
            .str.replace('%', '', regex=False)
            .replace({'N/A': np.nan, 'nan': np.nan, '': np.nan})
            .astype(float)
        )

bool_map = {'t': 1.0, 'f': 0.0}
for col in ['host_is_superhost', 'instant_bookable', 'host_identity_verified', 'host_has_profile_pic']:
    if col in meta.columns:
        meta[f'{col}_enc'] = meta[col].map(bool_map).astype('float32')

RESPONSE_TIME_MAP = {
    'within an hour': 4,
    'within a few hours': 3,
    'within a day': 2,
    'a few days or more': 1,
}
if 'host_response_time' in meta.columns:
    meta['host_response_time_enc'] = meta['host_response_time'].map(RESPONSE_TIME_MAP).astype('float32')
    meta['host_response_time_missing'] = meta['host_response_time'].isna().astype('int8')
    print(f'host_response_time_enc: {meta["host_response_time_enc"].notna().mean():.1%} coverage')

# NOTE: availability_30/60/90/365 excluded -- Airbnb's forward-looking counts from the
# Jan 2026 scrape date directly overlap the Feb-Apr 2026 prediction window (leakage).
NUMERIC_COLS = [
    'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'minimum_nights', 'maximum_nights',
    'minimum_minimum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm',
    'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d',
    'number_of_reviews_ly', 'reviews_per_month',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value',
    'estimated_occupancy_l365d', 'estimated_revenue_l365d',
    'calculated_host_listings_count',
    'hosts_time_as_host_years', 'hosts_time_as_host_months',
    'hosts_time_as_user_years', 'hosts_time_as_user_months',
]
for col in NUMERIC_COLS:
    if col in meta.columns:
        meta[col] = pd.to_numeric(meta[col], errors='coerce').astype('float32')

print('Numeric parsing done.')
print(f'  price_numeric: {meta["price_numeric"].notna().mean():.1%} coverage')

host_response_time_enc: 0.0% coverage
Numeric parsing done.
  price_numeric: 62.1% coverage


In [13]:
REF_DATE = pd.Timestamp('2026-02-01')  # start of Setup C's prediction window (Feb-Apr 2026)

for col in ['host_since', 'first_review', 'last_review']:
    if col in meta.columns:
        meta[col] = pd.to_datetime(meta[col], errors='coerce')

if 'host_since'   in meta.columns: meta['host_age_days']          = (REF_DATE - meta['host_since']).dt.days.astype('float32')
if 'first_review' in meta.columns: meta['listing_age_days']       = (REF_DATE - meta['first_review']).dt.days.astype('float32')
if 'last_review'  in meta.columns: meta['days_since_last_review'] = (REF_DATE - meta['last_review']).dt.days.astype('float32')

meta['has_reviews']    = (meta.get('number_of_reviews', pd.Series(0)) > 0).astype('int8')
if 'price_numeric' in meta.columns and 'accommodates' in meta.columns:
    meta['price_per_guest'] = (meta['price_numeric'] / meta['accommodates'].replace(0, np.nan)).astype('float32')
if 'price_numeric' in meta.columns and 'minimum_nights' in meta.columns:
    meta['min_stay_x_price'] = (meta['minimum_nights'] * meta['price_numeric']).astype('float32')
if 'estimated_revenue_l365d' in meta.columns and 'estimated_occupancy_l365d' in meta.columns:
    meta['implied_nightly_rate'] = (meta['estimated_revenue_l365d'] /
                                    meta['estimated_occupancy_l365d'].replace(0, np.nan)).astype('float32')

print('Date / derived features added.')
for c in ['host_age_days', 'listing_age_days', 'days_since_last_review']:
    if c in meta.columns:
        print(f'  {c}: {meta[c].notna().mean():.1%} coverage, median={meta[c].median():.0f} days')

Date / derived features added.
  host_age_days: 0.0% coverage, median=nan days
  listing_age_days: 68.8% coverage, median=1706 days
  days_since_last_review: 68.8% coverage, median=598 days


In [14]:
def parse_bathrooms(s):
    if pd.isna(s) or str(s).strip() == '': return np.nan, np.nan
    s = str(s).lower().strip()
    is_shared = 1.0 if 'shared' in s else 0.0
    m = re.search(r'(\d+\.?\d*)', s)
    num = float(m.group(1)) if m else np.nan
    if 'half' in s and (pd.isna(num) or num == 0): num = 0.5
    return num, is_shared

if 'bathrooms_text' in meta.columns:
    parsed = meta['bathrooms_text'].apply(parse_bathrooms)
    meta['bathrooms_num']  = parsed.apply(lambda x: x[0]).astype('float32')
    meta['bath_is_shared'] = parsed.apply(lambda x: x[1]).astype('float32')
    if 'bathrooms' in meta.columns:
        meta['bathrooms_num'] = meta['bathrooms_num'].fillna(meta['bathrooms'])
    print(f'bathrooms_num  : {meta["bathrooms_num"].notna().mean():.1%} coverage')
    print(f'bath_is_shared : {meta["bath_is_shared"].value_counts().to_dict()}')

bathrooms_num  : 99.8% coverage
bath_is_shared : {0.0: 26643, 1.0: 9534}


In [15]:
KEY_AMENITIES = {
    'wifi':         ['wifi', 'fast wifi'],
    'kitchen':      ['kitchen', 'kitchenette'],
    'ac':           ['air conditioning', 'central air conditioning'],
    'washer':       ['washer'],
    'dryer':        ['dryer'],
    'parking':      ['free parking', 'paid parking', 'free street parking', 'free driveway parking'],
    'elevator':     ['elevator'],
    'tv':           ['tv', 'hdtv'],
    'pool':         ['pool'],
    'gym':          ['gym', 'exercise equipment', 'fitness room'],
    'co_alarm':     ['carbon monoxide alarm'],
    'long_term':    ['long term stays allowed'],
    'self_checkin': ['self check-in', 'lockbox', 'smart lock', 'keypad'],
    'hot_tub':      ['hot tub', 'private hot tub'],
}

if 'amenities' in meta.columns:
    amen_str = meta['amenities'].fillna('').astype(str).str.lower()
    for flag, keywords in KEY_AMENITIES.items():
        meta[f'amen_{flag}'] = amen_str.apply(
            lambda s: int(any(kw in s for kw in keywords))
        ).astype('int8')
    print('Amenity flags coverage:')
    for flag in KEY_AMENITIES:
        print(f'  amen_{flag}: {meta[f"amen_{flag}"].mean():.1%}')

Amenity flags coverage:
  amen_wifi: 97.9%
  amen_kitchen: 88.6%
  amen_ac: 76.5%
  amen_washer: 50.5%
  amen_dryer: 73.3%
  amen_parking: 46.2%
  amen_elevator: 24.8%
  amen_tv: 78.2%
  amen_pool: 3.2%
  amen_gym: 18.1%
  amen_co_alarm: 78.7%
  amen_long_term: 35.7%
  amen_self_checkin: 39.8%
  amen_hot_tub: 4.8%


### 4.1 Title / Description Keyword Flags

In [16]:
name_s = meta['name'].fillna('').astype(str) if 'name' in meta.columns else pd.Series('', index=meta.index)
desc_s = meta['description'].fillna('').astype(str) if 'description' in meta.columns else pd.Series('', index=meta.index)
txt = (name_s + ' ' + desc_s).str.lower()

KEYWORD_PATTERNS = {
    'luxury':   r'luxury|upscale|fancy|premium',
    'private':  r'private|quiet|secluded|separate',
    'cozy':     r'cozy|charming|warm|homey',
    'modern':   r'modern|stylish|renovated|new',
    'view':     r'view|scenic|panorama',
    'near':     r'near|close to|steps to|minutes to',
    'spacious': r'spacious|roomy|large|huge',
    'studio':   r'studio',
}
for kw, pattern in KEYWORD_PATTERNS.items():
    meta[f'kw_{kw}'] = txt.str.contains(pattern, regex=True).astype('int8')

meta['title_len'] = name_s.str.len().astype('int32')
meta['title_words'] = name_s.str.split().str.len().fillna(0).astype('int32')

print('Keyword flags added:', [f'kw_{kw}' for kw in KEYWORD_PATTERNS])
for kw in KEYWORD_PATTERNS:
    print(f'  kw_{kw}: {meta[f"kw_{kw}"].mean():.1%}')
print(f'title_len   : median={meta["title_len"].median():.0f} chars')
print(f'title_words : median={meta["title_words"].median():.0f} words')

Keyword flags added: ['kw_luxury', 'kw_private', 'kw_cozy', 'kw_modern', 'kw_view', 'kw_near', 'kw_spacious', 'kw_studio']
  kw_luxury: 7.6%
  kw_private: 38.3%
  kw_cozy: 23.1%
  kw_modern: 43.6%
  kw_view: 11.7%
  kw_near: 32.4%
  kw_spacious: 30.1%
  kw_studio: 12.0%
title_len   : median=38 chars
title_words : median=6 words


In [17]:
def map_property_type(s):
    if pd.isna(s): return 'Other'
    s = str(s).lower()
    if 'hotel' in s or 'hostel' in s or 'boutique' in s: return 'Hotel_Hostel'
    if s.startswith('entire'): return 'Entire_Place'
    if 'private room' in s: return 'Private_Room'
    if 'shared room' in s: return 'Shared_Room'
    return 'Other'

if 'property_type' in meta.columns:
    meta['property_type_grp'] = meta['property_type'].apply(map_property_type)

for col in ['property_type_grp', 'room_type', 'neighbourhood_group_cleansed']:
    if col in meta.columns:
        dummies = pd.get_dummies(meta[col], prefix=col, drop_first=True, dtype='int8')
        meta = pd.concat([meta, dummies], axis=1)
        print(f'{col}: {dummies.shape[1]} dummy columns added | categories: {meta[col].unique().tolist()}')

if 'neighbourhood_cleansed' in meta.columns:
    nb_freq = meta['neighbourhood_cleansed'].value_counts()
    meta['neighbourhood_freq'] = meta['neighbourhood_cleansed'].map(nb_freq).astype('float32')
    print(f'neighbourhood_freq: {meta["neighbourhood_freq"].describe().round(0).to_dict()}')

property_type_grp: 4 dummy columns added | categories: ['Private_Room', 'Entire_Place', 'Hotel_Hostel', 'Shared_Room', 'Other']
room_type: 3 dummy columns added | categories: ['Private room', 'Entire home/apt', 'Hotel room', 'Shared room']
neighbourhood_group_cleansed: 4 dummy columns added | categories: ['Brooklyn', 'Manhattan', 'Bronx', 'Queens', 'Staten Island']
neighbourhood_freq: {'count': 36282.0, 'mean': 972.0, 'std': 806.0, 'min': 1.0, '25%': 264.0, '50%': 634.0, '75%': 1525.0, 'max': 2618.0}


In [18]:
if 'latitude' in meta.columns and 'longitude' in meta.columns:
    coords = meta[['latitude', 'longitude']].copy()
    coords_filled = coords.fillna({'latitude': coords['latitude'].median(),
                                   'longitude': coords['longitude'].median()})
    kmeans = KMeans(n_clusters=20, random_state=RANDOM_STATE, n_init=10)
    meta['geo_cluster'] = kmeans.fit_predict(coords_filled).astype('int8')
    print(f'KMeans geo cluster: 20 clusters')
    print(f'Cluster sizes (top 5): {meta["geo_cluster"].value_counts().head(5).to_dict()}')

    geo_dummies = pd.get_dummies(meta['geo_cluster'], prefix='geo_cluster', drop_first=True, dtype='int8')
    meta = pd.concat([meta, geo_dummies], axis=1)
    print(f'geo_cluster: {geo_dummies.shape[1]} dummy columns added')

KMeans geo cluster: 20 clusters
Cluster sizes (top 5): {1: 5585, 4: 4500, 11: 3182, 14: 3027, 8: 2856}
geo_cluster: 19 dummy columns added


## 5. Review Velocity (January 2026 reviews, cutoff Jan 31 2026)

In [19]:
print('Loading January 2026 reviews...')
reviews = pd.read_csv(
    JAN_DIR / 'reviews.csv',
    usecols=['listing_id', 'date'],
    dtype={'listing_id': 'int64'},
)
reviews['date'] = pd.to_datetime(reviews['date'], errors='coerce')
reviews = reviews.dropna(subset=['date'])
print(f'Reviews: {len(reviews):,} rows | date range: {reviews["date"].min().date()} -> {reviews["date"].max().date()}')

CUT_DATE = pd.Timestamp('2026-01-31')
print(f'\nReview velocity (as of {CUT_DATE.date()}):')
for days, col in [(30, 'rev_30d'), (90, 'rev_90d'), (180, 'rev_180d'), (365, 'rev_365d')]:
    start = CUT_DATE - pd.Timedelta(days=days)
    mask  = (reviews['date'] >= start) & (reviews['date'] <= CUT_DATE)
    counts = reviews[mask].groupby('listing_id').size().rename(col)
    meta[col] = (
        meta['id'].map(counts.rename(index=str))
        .fillna(0).astype('int32')
    )
    n_active = (meta[col] > 0).sum()
    print(f'  {col}: {n_active:,} listings with >=1 review  |  median={meta[col].median():.0f}')

meta['rev_accel_90'] = (meta['rev_90d'] - (meta['rev_180d'] - meta['rev_90d'])).astype('int32')
print(f'\n  rev_accel_90: mean={meta["rev_accel_90"].mean():.2f}  '
      f'(positive={ (meta["rev_accel_90"] > 0).mean():.1%}  negative={ (meta["rev_accel_90"] < 0).mean():.1%})')

Loading January 2026 reviews...


Reviews: 993,180 rows | date range: 2009-05-25 -> 2026-01-15

Review velocity (as of 2026-01-31):
  rev_30d: 1,901 listings with >=1 review  |  median=0
  rev_90d: 6,163 listings with >=1 review  |  median=0
  rev_180d: 8,723 listings with >=1 review  |  median=0
  rev_365d: 10,714 listings with >=1 review  |  median=0

  rev_accel_90: mean=-0.27  (positive=6.8%  negative=14.1%)


## 6. Derive Target — Clean Multi-Snapshot Rule (Target: Feb-Apr 2026)

Identical rule to Setup B: a date counts as booked only if observed >=2 times, was available
at some point, and unavailable at the last observation. Re-pointed at Setup C's own 6
snapshots: the last 3 history months (Nov 2025, Dec 2025, Jan 2026) plus the 3 target months
(Feb, Mar, Apr 2026) -- all already on disk, Apr 2026 confirmed present.

In [20]:
FEB_DIR = Q1_DIR / 'February2026'
MAR_DIR = Q1_DIR / 'March2026'
APR_DIR = Q1_DIR / 'April2026'

# order = scrape sequence; each entry observes Feb-Apr 2026 dates as a future date
TARGET_SNAPS = [
    (1, NOV_DIR / 'calendar.csv'),
    (2, DEC_DIR / 'calendar.csv'),
    (3, JAN_DIR / 'calendar.csv'),
    (4, FEB_DIR / 'calendar.csv'),
    (5, MAR_DIR / 'calendar.csv'),
    (6, APR_DIR / 'calendar.csv'),
]
TARGET_MONTHS = ('2026-02', '2026-03', '2026-04')
TARGET_CLIP = 89  # Feb(28, 2026 not leap) + Mar(31) + Apr(30)

print('Building clean Feb-Apr 2026 target from 6 monthly snapshots (Nov25-Apr26)...')
snap_frames = []
for order, cal_path in TARGET_SNAPS:
    df = pd.read_csv(cal_path, usecols=['listing_id', 'date', 'available'],
                      dtype={'listing_id': 'int64', 'available': 'category'})
    df = df[df['date'].str.slice(0, 7).isin(TARGET_MONTHS)].copy()
    df['order'] = np.int8(order)
    df['av'] = (df['available'] == 't')
    snap_frames.append(df[['listing_id', 'date', 'order', 'av']])
    print(f'  snapshot {order} ({cal_path.parent.name}): {len(df):,} Feb-Apr-2026 rows')

panel = pd.concat(snap_frames, ignore_index=True).sort_values(['listing_id', 'date', 'order'])
del snap_frames

# naive reference: January-2026 scrape's lone forward view (last pre-target snapshot, order==3)
jan_rows = panel[panel['order'] == 3]
y_naive = (~jan_rows['av']).astype('int8').groupby(jan_rows['listing_id']).sum()

g = panel.groupby(['listing_id', 'date'])['av']
trans = pd.DataFrame({'n_obs': g.size(), 'ever_avail': g.max(), 'last_av': g.last()})
trans['booked_clean'] = ((trans['n_obs'] >= 2) & trans['ever_avail'] & (~trans['last_av'])).astype('int8')
y_clean = trans.groupby(level='listing_id')['booked_clean'].sum()
del panel, trans

# reindex onto the January-2026 listing universe (this setup's base snapshot)
univ = meta[['id']].copy()
univ['listing_id'] = univ['id'].astype('int64')
target = univ.merge(y_clean.rename('blocked_days_Q1_2026'), on='listing_id', how='left')
target = target.merge(y_naive.rename('blocked_days_Q1_2026_naive'), on='listing_id', how='left')
target['blocked_days_Q1_2026'] = target['blocked_days_Q1_2026'].fillna(0).clip(0, TARGET_CLIP).astype('int32')
target['blocked_days_Q1_2026_naive'] = target['blocked_days_Q1_2026_naive'].fillna(0).clip(0, TARGET_CLIP).astype('int32')

y, yn = target['blocked_days_Q1_2026'], target['blocked_days_Q1_2026_naive']
print(f'\nTarget listings: {len(target):,}')
print(f'  clean (used) : mean={y.mean():.1f}  std={y.std():.1f}  var={y.var():.0f}  zero-rate={(y==0).mean():.3f}  full({TARGET_CLIP}d)-rate={(y==TARGET_CLIP).mean():.3f}')
print(f'  naive (ref)  : mean={yn.mean():.1f}  std={yn.std():.1f}  var={yn.var():.0f}  zero-rate={(yn==0).mean():.3f}  full({TARGET_CLIP}d)-rate={(yn==TARGET_CLIP).mean():.3f}')
print(f'  naive/clean variance ratio: {yn.var() / y.var():.2f}x')

target[['id', 'blocked_days_Q1_2026_naive']].rename(
    columns={'blocked_days_Q1_2026_naive': 'blocked_days_naive'}
).to_parquet(OUT_DIR / 'target_naive_setupC.parquet', index=False)
print('Saved target_naive_setupC.parquet')

Building clean Feb-Apr 2026 target from 6 monthly snapshots (Nov25-Apr26)...


  snapshot 1 (November2025): 3,235,417 Feb-Apr-2026 rows


  snapshot 2 (December2025): 3,227,229 Feb-Apr-2026 rows


  snapshot 3 (January2026): 3,229,454 Feb-Apr-2026 rows


  snapshot 4 (February2026): 2,780,754 Feb-Apr-2026 rows


  snapshot 5 (March2026): 1,635,936 Feb-Apr-2026 rows


  snapshot 6 (April2026): 564,263 Feb-Apr-2026 rows



Target listings: 36,282
  clean (used) : mean=17.9  std=26.4  var=697  zero-rate=0.486  full(89d)-rate=0.030
  naive (ref)  : mean=45.4  std=40.4  var=1630  zero-rate=0.316  full(89d)-rate=0.405
  naive/clean variance ratio: 2.34x
Saved target_naive_setupC.parquet


## 7. Merge All Features

In [21]:
for df in [agg_q3, agg_q4, agg_hist, agg_monthly, agg_recent, agg_vol]:
    if 'id' not in df.columns:
        df['id'] = df['listing_id'].astype(str)

# Start from target
feat = target[['id', 'blocked_days_Q1_2026']].copy()
print(f'Start: {feat.shape}')

for agg_df, name in [
    (agg_q3,      'Q3 calendar'),
    (agg_q4,      'Q4 calendar'),
    (agg_hist,    '7-month combined'),
    (agg_monthly, 'monthly breakdown'),
    (agg_recent,  'recency window'),
    (agg_vol,     'cross-snapshot volatility'),
]:
    merge_cols = [c for c in agg_df.columns if c not in ('listing_id', 'id')]
    feat = feat.merge(agg_df[['id'] + merge_cols], on='id', how='left')
    print(f'After {name}: {feat.shape}')

# Merge property metadata -- same drop list as Setup B
DROP_META = [
    'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description',
    'neighborhood_overview', 'picture_url', 'host_url', 'host_thumbnail_url',
    'host_picture_url', 'host_about', 'host_location', 'host_name',
    'host_since', 'host_verifications', 'host_neighbourhood',
    'amenities', 'price', 'bathrooms_text', 'bathrooms',
    'host_response_rate', 'host_acceptance_rate',
    'host_is_superhost', 'instant_bookable',
    'host_identity_verified', 'host_has_profile_pic',
    'first_review', 'last_review', 'host_since',
    'host_profile_id', 'host_profile_url', 'host_id',
    'property_type', 'property_type_grp',
    'room_type', 'neighbourhood',
    'neighbourhood_group_cleansed',
    'calendar_last_scraped', 'calendar_updated',
    'has_availability', 'availability_eoy',
    'host_response_time',
    'license',
    'availability_30', 'availability_60', 'availability_90', 'availability_365',
]
meta_cols = [c for c in meta.columns if c not in DROP_META]
feat = feat.merge(meta[meta_cols], on='id', how='left')
print(f'After metadata  : {feat.shape}')

Start: (36282, 2)
After Q3 calendar: (36282, 14)


After Q4 calendar: (36282, 26)
After 7-month combined: (36282, 38)
After monthly breakdown: (36282, 52)
After recency window: (36282, 64)
After cross-snapshot volatility: (36282, 67)
After metadata  : (36282, 195)


In [22]:
# Cross-table derived features -- Q3/Q4 split still exists for Setup C, unchanged from Setup B.
if 'q3_blocking_rate' in feat.columns and 'q4_blocking_rate' in feat.columns:
    feat['q3_to_q4_blocking_trend'] = (feat['q4_blocking_rate'] - feat['q3_blocking_rate']).astype('float32')
    feat['q3_to_q4_blocking_ratio'] = (
        feat['q4_blocking_rate'] / feat['q3_blocking_rate'].replace(0, np.nan)
    ).astype('float32')
    print('q3_to_q4_blocking_trend added (Q4 - Q3 blocking rate)')
    print('q3_to_q4_blocking_ratio added (Q4 / Q3 blocking rate)')

if 'recent_blocking_rate' in feat.columns:
    feat['recent_fully_blocked'] = (feat['recent_blocking_rate'] >= 1.0).astype('int8')
    print(f'recent_fully_blocked added: {feat["recent_fully_blocked"].mean():.1%} of listings')

print(f'\nFinal feature matrix: {feat.shape}')

q3_to_q4_blocking_trend added (Q4 - Q3 blocking rate)
q3_to_q4_blocking_ratio added (Q4 / Q3 blocking rate)
recent_fully_blocked added: 52.0% of listings

Final feature matrix: (36282, 198)


### 7.1 Trajectory (Monthly Trend) Features

7 monthly points now (M7...M12, M1) instead of Setup B's 6. `recent3` = the last 3
chronological months (Nov, Dec, Jan); `older3` = the first 3 (Jul, Aug, Sep); October (the
middle point) is not used in either bucket -- the same kind of one-point-left-over trade-off
Setup B doesn't have to make, an unavoidable consequence of splitting an odd-length (7) series
into two groups of 3.

In [23]:
mcols = ['m7_blocking_rate', 'm8_blocking_rate', 'm9_blocking_rate', 'm10_blocking_rate',
         'm11_blocking_rate', 'm12_blocking_rate', 'm1_blocking_rate']
M = feat[mcols].fillna(0).to_numpy(dtype=float)
t = np.arange(7); tc = t - t.mean()

feat['traj_slope'] = (M * tc).sum(1) / (tc ** 2).sum()
feat['traj_recent3'] = M[:, 4:].mean(1)   # Nov, Dec, Jan
feat['traj_older3']  = M[:, :3].mean(1)   # Jul, Aug, Sep
feat['traj_recent_older_ratio'] = feat['traj_recent3'] / (feat['traj_older3'] + 0.05)
feat['traj_accel'] = M[:, 6] - M[:, 5]    # Jan - Dec
feat['traj_std'] = M.std(1)
feat['traj_nonzero_months'] = (M > 0).sum(1).astype('int8')

traj_cols = ['traj_slope', 'traj_recent3', 'traj_older3', 'traj_recent_older_ratio',
             'traj_accel', 'traj_std', 'traj_nonzero_months']
print('Trajectory features added:', traj_cols)
print(feat[traj_cols].describe().round(3).to_string())

Trajectory features added: ['traj_slope', 'traj_recent3', 'traj_older3', 'traj_recent_older_ratio', 'traj_accel', 'traj_std', 'traj_nonzero_months']
       traj_slope  traj_recent3  traj_older3  traj_recent_older_ratio  traj_accel   traj_std  traj_nonzero_months
count   36282.000     36282.000    36282.000                36282.000   36282.000  36282.000            36282.000
mean       -0.003         0.627        0.637                    1.284      -0.028      0.147                5.686
std         0.060         0.394        0.399                    2.339       0.343      0.167                2.229
min        -0.214         0.000        0.000                    0.000      -1.000      0.000                0.000
25%        -0.006         0.255        0.236                    0.592       0.000      0.000                5.000
50%         0.000         0.747        0.808                    0.952       0.000      0.041                7.000
75%         0.006         1.000        1.000         

### 7.2 Neighbourhood / Geo-Cluster Context Features

In [24]:
group_rate_col = 'recent_blocking_rate'  # January 2026 window -- closest-to-target history signal

for key, prefix in [('neighbourhood_cleansed', 'nb'), ('geo_cluster', 'geo')]:
    if key not in feat.columns:
        print(f'{key} not found, skipping'); continue
    feat[f'{prefix}_mean_rate'] = feat.groupby(key)[group_rate_col].transform('mean')
    feat[f'{prefix}_density']   = feat.groupby(key)[group_rate_col].transform('size')
    feat[f'rate_vs_{prefix}']   = feat[group_rate_col].fillna(0) - feat[f'{prefix}_mean_rate']
    print(f'{prefix}_mean_rate / {prefix}_density / rate_vs_{prefix} added (grouped by {key})')

feat = feat.drop(columns=['geo_cluster'])
print(f'\nFinal feature matrix after context features: {feat.shape}')

nb_mean_rate / nb_density / rate_vs_nb added (grouped by neighbourhood_cleansed)
geo_mean_rate / geo_density / rate_vs_geo added (grouped by geo_cluster)

Final feature matrix after context features: (36282, 210)


## 8. Feature Summary & Correlations

In [25]:
n_num = feat.select_dtypes(include='number').shape[1]
n_obj = feat.select_dtypes(include='object').shape[1]
obj_remaining = feat.select_dtypes(include='object').columns.tolist()
print(f'Feature matrix: {feat.shape}')
print(f'  Numeric  : {n_num}')
print(f'  Object   : {n_obj}  -> {obj_remaining}')

mv = feat.isna().mean().sort_values(ascending=False)
mv_top = mv[mv > 0].head(25)
print(f'\nTop columns by missing rate ({len(mv[mv>0])} columns have NaN):')
print((mv_top * 100).round(1).astype(str).to_string())

Feature matrix: (36282, 210)
  Numeric  : 208
  Object   : 2  -> ['id', 'neighbourhood_cleansed']

Top columns by missing rate (88 columns have NaN):
instant_bookable_enc         100.0
host_response_time_enc       100.0
implied_nightly_rate         100.0
estimated_revenue_l365d      100.0
host_response_rate_num       100.0
host_acceptance_rate_num     100.0
host_total_listings_count    100.0
price_implied                100.0
host_age_days                100.0
p5_trend                      54.8
p5_cv                         41.4
p5_std                        41.4
beds                          40.1
min_stay_x_price              37.9
price_numeric                 37.9
price_per_guest               37.9
price_rel_nbhd                37.9
p5_mean                       37.9
p5_max                        37.9
p5_min                        37.9
p5_range                      37.9
review_scores_location        31.2
review_scores_value           31.2
review_scores_checkin         31.2
review_sco

In [26]:
num_feat = feat.select_dtypes(include='number').copy()
corr_target = (
    num_feat.corr()['blocked_days_Q1_2026']
    .drop('blocked_days_Q1_2026', errors='ignore')
    .sort_values(key=abs, ascending=False)
)
print('Top 30 features by |Pearson r| with blocked_days_Q1_2026 (semantically: Feb-Apr 2026 target):')
print(corr_target.head(30).round(4).to_string())

Top 30 features by |Pearson r| with blocked_days_Q1_2026 (semantically: Feb-Apr 2026 target):
hist_flip_rate                                 0.5101
hist_flips                                     0.4861
days_since_last_review                        -0.4743
p5_n_distinct                                  0.4518
traj_std                                       0.4290
price_missing                                 -0.4225
estimated_occupancy_l365d                      0.3752
host_is_superhost_enc                          0.3182
m11_days                                       0.3024
listing_age_days                              -0.2661
amen_long_term                                 0.2530
q3_new_unblockings                             0.2432
hist_new_blockings                             0.2421
hist_new_unblockings                           0.2396
q3_new_blockings                               0.2393
q4_new_unblockings                             0.2345
amen_self_checkin                         

## 9. Train / Test Split & Save

In [27]:
KEEP_RAW_CAT = ['neighbourhood_cleansed']
obj_cols = feat.select_dtypes(include='object').columns.tolist()
obj_cols_drop = [c for c in obj_cols if c != 'id' and c not in KEEP_RAW_CAT]
feat_model = feat.drop(columns=obj_cols_drop)
print(f'Object columns dropped: {obj_cols_drop}')
print(f'Object columns kept (raw categorical): {[c for c in KEEP_RAW_CAT if c in feat_model.columns]}')
print(f'Model-ready matrix: {feat_model.shape}')

train_local, test_local = train_test_split(
    feat_model, test_size=0.20, random_state=RANDOM_STATE, shuffle=True
)
print(f'\nTrain (80%): {train_local.shape}')
print(f'Test  (20%): {test_local.shape}')

print(f'\nTrain target -- mean={train_local["blocked_days_Q1_2026"].mean():.2f}  '
      f'std={train_local["blocked_days_Q1_2026"].std():.2f}')
print(f'Test  target -- mean={test_local["blocked_days_Q1_2026"].mean():.2f}  '
      f'std={test_local["blocked_days_Q1_2026"].std():.2f}')

Object columns dropped: []
Object columns kept (raw categorical): ['neighbourhood_cleansed']
Model-ready matrix: (36282, 210)

Train (80%): (29025, 210)
Test  (20%): (7257, 210)

Train target -- mean=17.94  std=26.44
Test  target -- mean=17.78  std=26.22


In [28]:
try:
    train_local.to_parquet(OUT_DIR / 'train_local_setupC.parquet',  index=False)
    test_local .to_parquet(OUT_DIR / 'test_local_setupC.parquet',   index=False)
    feat_model .to_parquet(OUT_DIR / 'all_features_setupC.parquet', index=False)
    print('Saved parquet files to', OUT_DIR.resolve())
except Exception as e:
    print('Parquet failed, saving CSV:', e)
    train_local.to_csv(OUT_DIR / 'train_local_setupC.csv',  index=False)
    test_local .to_csv(OUT_DIR / 'test_local_setupC.csv',   index=False)
    feat_model .to_csv(OUT_DIR / 'all_features_setupC.csv', index=False)

print('\nSetup C files saved:')
for f in OUT_DIR.glob('*setupC*'):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.2f} MB)')

Saved parquet files to /Users/bashkal/Desktop/ML/ML-Final/Internship/outputs

Setup C files saved:
  train_local_setupC.parquet  (5.27 MB)
  target_naive_setupC.parquet  (0.47 MB)
  all_features_setupC.parquet  (6.08 MB)
  test_local_setupC.parquet  (1.56 MB)


## Summary

Setup C (Feb-Apr 2026 target, Jul 2025-Jan 2026 history) feature matrix built from the January
2026 listings snapshot + the same 6-snapshot clean multi-snapshot target rule used in Setup B,
re-pointed at Nov 2025-Apr 2026 calendars. Superset of Setup B's feature groups (adds a 7th
monthly block and a 7-point trajectory series) since the history window is longer. Consumed by
`16_Cross_Period_Robustness.ipynb` and `17_Ablation_and_Hurdle_Cross_Period.ipynb`.